# Tennis Match Prediction Chatbot

An independent Python machine-learning project that combines tennis data collection,
feature engineering, logistic regression, model evaluation, and an interactive
command-line chatbot.

> **Important limitation:** the notebook includes fallback simulated data when the
> configured historical dataset cannot provide the statistics required by the model.
> See the README for details.


## 1. Setup

The project uses Pandas and NumPy for data processing, Requests and BeautifulSoup
for data collection, and scikit-learn for model training and evaluation.


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import numpy as np
import requests
from bs4 import BeautifulSoup
import time
import io
import random
import re


## 2. Player-name normalization

Normalizes player names so scraped rankings and historical records can be matched more consistently.


In [ ]:
# --- Helper Function for Player Name Normalization ---
def _normalize_player_name(name: str) -> str:
    """Normalizes player names for consistent matching (lowercase, strip, basic cleanup)."""
    name = name.lower().strip()
    # Remove any parenthetical additions (like " (ATP)")
    name = re.sub(r'\s+\(.*?\)$', '', name)
    # Remove leading/trailing periods, often seen with initials in some datasets
    name = name.strip('.')
    # Consider replacing common special characters or multiple spaces with single spaces
    name = re.sub(r'\s+', ' ', name)
    return name


## 3. ATP rankings scraper

Retrieves ATP singles rankings, caches results, and uses a fallback player list if scraping fails.


In [ ]:
# --- 0. Web Scraping for Live Player Rankings ---
class ATPScraper:
    def __init__(self):
        self.rankings_url = "https://www.atptour.com/en/rankings/singles"
        self.player_rankings = [] # Stores normalized names
        self.last_scrape_time = 0 # To prevent overly frequent scraping

    def _fetch_html(self, url: str) -> str:
        """Fetches URL content with delay and error handling."""
        try:
            time.sleep(1) # Be polite
            response = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=10)
            response.raise_for_status()
            return response.text
        except requests.exceptions.RequestException as e:
            print(f"Error fetching {url}: {e}")
            return ""

    def scrape_player_rankings(self, num_players: int = 100, force_scrape: bool = False) -> list:
        """Scrapes top N player names and ranks. Caches results."""
        # Use cached rankings if available and not forced to scrape
        if not force_scrape and self.player_rankings and (time.time() - self.last_scrape_time < 3600):
            return self.player_rankings

        print(f"Scraping top {num_players} player rankings from ATP Tour...")
        html = self._fetch_html(self.rankings_url)

        scraped_players = []
        if html:
            soup = BeautifulSoup(html, 'html.parser')
            rows = soup.select('table.ranking-table tbody tr')
            for row in rows:
                if len(scraped_players) >= num_players: break
                try:
                    rank = int(row.select_one('td.rank-cell').get_text(strip=True))
                    name_element = row.select_one('td.player-cell a')
                    original_name = name_element.get_text(strip=True) if name_element else row.select_one('td.player-cell').get_text(strip=True)
                    normalized_name = _normalize_player_name(original_name)
                    scraped_players.append({"name": normalized_name, "original_name": original_name, "rank": rank})
                except (ValueError, AttributeError):
                    # Skip rows that don't conform to expected structure
                    continue

        if not scraped_players:
            print("Failed to scrape any rankings. Using fallback players.")
            fallback_players_raw = [
                "Novak Djokovic", "Carlos Alcaraz", "Jannik Sinner", "Daniil Medvedev",
                "Alexander Zverev", "Andrey Rublev", "Casper Ruud", "Hubert Hurkacz",
                "Taylor Fritz", "Alex de Minaur", "Grigor Dimitrov", "Stefanos Tsitsipas",
                "Tommy Paul", "Ben Shelton", "Frances Tiafoe", "Holger Rune",
                "Felix Auger-Aliassime", "Karen Khachanov", "Lorenzo Musetti", "Cameron Norrie",
                "Denis Shapovalov", "Diego Schwartzman", "Gael Monfils", "Andy Murray",
                "Stan Wawrinka", "Richard Gasquet", "Fabio Fognini", "Marin Cilic",
                "John Isner", "Kevin Anderson", "Juan Martin del Potro", "Nick Kyrgios"
            ]
            self.player_rankings = [{"name": _normalize_player_name(p), "original_name": p, "rank": i+1} for i, p in enumerate(fallback_players_raw)]
        else:
            self.player_rankings = scraped_players

        print(f"Scraped {len(self.player_rankings)} player rankings (live or fallback).")
        self.last_scrape_time = time.time()
        return self.player_rankings

    def get_player_names(self) -> list:
        # This function should return user-friendly names, not necessarily the normalized ones.
        if not self.player_rankings:
            self.scrape_player_rankings()
        return sorted(list(set([p['original_name'] for p in self.player_rankings]))) # Return original names for display

    def get_player_rank(self, player_name: str) -> int | None:
        if not self.player_rankings or (time.time() - self.last_scrape_time > 3600):
            self.scrape_player_rankings()

        normalized_player_name = _normalize_player_name(player_name)

        # Try exact match first
        for p in self.player_rankings:
            if p['name'] == normalized_player_name:
                return p['rank']

        # If no exact match, try partial matching (e.g., "djokovic" matches "novak djokovic")
        for p in self.player_rankings:
            if normalized_player_name in p['name'] or p['name'] in normalized_player_name:
                return p['rank']

        return None # No rank found


## 4. Historical match data and player statistics

Loads match data, prepares player-level statistics, and includes fallback behavior when required fields are unavailable.


In [ ]:
# --- 1. Data Acquisition and Aggregation ---
class TennisDataLoader:
    # Use a list of URLs for multiple years of data
    ATP_MATCH_DATA_URLS = [
        "https://raw.githubusercontent.com/JeffSackmann/tennis_atp/master/atp_matches_2024.csv",

    ]

    def __init__(self, atp_scraper: ATPScraper):
        self.atp_scraper = atp_scraper
        self.match_data = self._load_historical_match_data()
        self.player_stats_cache = {} # Keyed by normalized player name

    def _load_historical_match_data(self) -> pd.DataFrame:
        """Loads historical match data from multiple URLs or falls back to simulation."""
        all_dfs = []
        for url in self.ATP_MATCH_DATA_URLS:
            print(f"Attempting to load historical match data from: {url}")
            try:
                response = requests.get(url, timeout=15)
                response.raise_for_status()
                df = pd.read_csv(io.StringIO(response.text), encoding='utf-8', on_bad_lines='skip', low_memory=False)
                print(f"Loaded {len(df)} matches from {url}.")
                all_dfs.append(df)
            except requests.exceptions.RequestException as e:
                print(f"Network or HTTP error loading data from {url}: {e}. Skipping this year.")
            except pd.errors.ParserError as e:
                print(f"CSV parsing error with {url}: {e}. Skipping this year.")
            except Exception as e:
                print(f"An unexpected error occurred processing data from {url}: {e}. Skipping this year.")

        if not all_dfs:
            print("Failed to load any historical match data from provided URLs. Falling back to simulated data.")
            return self._generate_simulated_match_data_fallback()

        df_combined = pd.concat(all_dfs, ignore_index=True)
        print(f"Loaded a total of {len(df_combined)} historical matches across all available years.")

        # Normalize player names immediately upon loading
        df_combined['winner_name'] = df_combined['winner_name'].apply(_normalize_player_name)
        df_combined['loser_name'] = df_combined['loser_name'].apply(_normalize_player_name)

        expected_cols = {
            'w_1stIn', 'w_1stWon', 'w_2ndWon', 'w_svpt', 'w_aces', 'w_ue',
            'l_1stIn', 'l_1stWon', 'l_2ndWon', 'l_svpt', 'l_aces', 'l_ue'
        }

        # Check for critical columns AFTER loading all data
        missing_critical_cols = not all(col in df_combined.columns for col in expected_cols)

        if missing_critical_cols:
            print("Warning: Key columns (e.g., serve stats, unforced errors) missing from combined historical data. Generating simulated data.")
            return self._generate_simulated_match_data_fallback()

        df_combined = df_combined.rename(columns={
            'w_1stIn': 'winner_1stIn', 'w_1stWon': 'winner_1stWon', 'w_2ndWon': 'winner_2ndWon',
            'l_1stIn': 'loser_1stIn', 'l_1stWon': 'loser_1stWon', 'l_2ndWon': 'loser_2ndWon',
            'w_svpt': 'winner_svpt', 'l_svpt': 'loser_svpt',
            'w_aces': 'winner_aces', 'l_aces': 'loser_aces',
            'w_ue': 'winner_unforced_errors', 'l_ue': 'loser_unforced_errors'
        })

        for prefix in ['winner', 'loser']:
            df_combined[f'{prefix}_first_serve_percentage'] = df_combined[f'{prefix}_1stIn'].replace(0, np.nan) / df_combined[f'{prefix}_svpt'].replace(0, np.nan)
            df_combined[f'{prefix}_first_serve_points_won_percentage'] = df_combined[f'{prefix}_1stWon'].replace(0, np.nan) / df_combined[f'{prefix}_1stIn'].replace(0, np.nan)
            df_combined[f'{prefix}_second_serve_points_won_percentage'] = df_combined[f'{prefix}_2ndWon'].replace(0, np.nan) / (df_combined[f'{prefix}_svpt'] - df_combined[f'{prefix}_1stIn']).replace(0, np.nan)

            opponent_prefix = 'loser' if prefix == 'winner' else 'winner'
            opponent_total_serve_points = df_combined[f'{opponent_prefix}_svpt'].fillna(0)
            df_combined[f'{prefix}_return_points_won_percentage'] = (
                (opponent_total_serve_points - (df_combined[f'{opponent_prefix}_1stWon'].fillna(0) + df_combined[f'{opponent_prefix}_2ndWon'].fillna(0))) /
                opponent_total_serve_points.replace(0, np.nan)
            )

        df_combined = df_combined.fillna(0)
        return df_combined

    def _generate_simulated_match_data_fallback(self, num_matches: int = 10000) -> pd.DataFrame:
        """Generates simulated historical match data as fallback."""
        print(f"Generating fallback simulated historical match data for {num_matches} matches...")
        players_for_sim = self.atp_scraper.get_player_names() # Get original names for simulation population
        if not players_for_sim:
            players_for_sim = [ # These will be normalized below
                "Novak Djokovic", "Carlos Alcaraz", "Jannik Sinner", "Daniil Medvedev",
                "Alexander Zverev", "Andrey Rublev", "Casper Ruud", "Hubert Hurkacz",
                "Taylor Fritz", "Alex de Minaur", "Grigor Dimitrov", "Stefanos Tsitsipas",
                "Tommy Paul", "Ben Shelton", "Frances Tiafoe", "Holger Rune",
                "Felix Auger-Aliassime", "Karen Khachanov", "Lorenzo Musetti", "Cameron Norrie",
                "Denis Shapovalov", "Diego Schwartzman", "Gael Monfils", "Andy Murray",
                "Stan Wawrinka", "Richard Gasquet", "Fabio Fognini", "Marin Cilic",
                "John Isner", "Kevin Anderson", "Juan Martin del Potro", "Nick Kyrgios",
                "Dominic Thiem", "Kei Nishikori", "Marin Cilic", "David Goffin", "Milos Raonic"
            ]

        data = []
        for i in range(num_matches):
            p1_display, p2_display = random.sample(players_for_sim, 2)
            p1 = _normalize_player_name(p1_display)
            p2 = _normalize_player_name(p2_display)

            p1_fs_perc = np.random.uniform(0.55, 0.75)
            p1_fs_pts_won_perc = np.random.uniform(0.65, 0.85)
            p1_ss_pts_won_perc = np.random.uniform(0.40, 0.60)
            p1_ret_pts_won_perc = np.random.uniform(0.30, 0.45)
            p1_aces = np.random.randint(4, 20)
            p1_ue = np.random.randint(15, 40)

            p2_fs_perc = np.random.uniform(0.55, 0.75)
            p2_fs_pts_won_perc = np.random.uniform(0.65, 0.85)
            p2_ss_pts_won_perc = np.random.uniform(0.40, 0.60)
            p2_ret_pts_won_perc = np.random.uniform(0.30, 0.45)
            p2_aces = np.random.randint(4, 20)
            p2_ue = np.random.randint(15, 40)

            p1_strength = (p1_fs_pts_won_perc * 0.35 + p1_ss_pts_won_perc * 0.15 + p1_ret_pts_won_perc * 0.30 - p1_ue * 0.003 + p1_aces * 0.002)
            p2_strength = (p2_fs_pts_won_perc * 0.35 + p2_ss_pts_won_perc * 0.15 + p2_ret_pts_won_perc * 0.30 - p2_ue * 0.003 + p2_aces * 0.002)

            if p1_strength + np.random.uniform(-0.08, 0.08) > p2_strength + np.random.uniform(-0.08, 0.08):
                winner, loser = p1, p2
                winner_stats = {'fs_perc': p1_fs_perc, 'fs_pts_won_perc': p1_fs_pts_won_perc, 'ss_pts_won_perc': p1_ss_pts_won_perc, 'ret_pts_won_perc': p1_ret_pts_won_perc, 'aces': p1_aces, 'ue': p1_ue}
                loser_stats = {'fs_perc': p2_fs_perc, 'fs_pts_won_perc': p2_fs_pts_won_perc, 'ss_pts_won_perc': p2_ss_pts_won_perc, 'ret_pts_won_perc': p2_ret_pts_won_perc, 'aces': p2_aces, 'ue': p2_ue}
            else:
                winner, loser = p2, p1
                winner_stats = {'fs_perc': p2_fs_perc, 'fs_pts_won_perc': p2_fs_pts_won_perc, 'ss_pts_won_perc': p2_ss_pts_won_perc, 'ret_pts_won_perc': p2_ret_pts_won_perc, 'aces': p2_aces, 'ue': p2_ue}
                loser_stats = {'fs_perc': p1_fs_perc, 'fs_pts_won_perc': p1_fs_pts_won_perc, 'ss_pts_won_perc': p1_ss_pts_won_perc, 'ret_pts_won_perc': p1_ret_pts_won_perc, 'aces': p1_aces, 'ue': p1_ue}

            data.append({
                "winner_name": winner, "loser_name": loser,
                "winner_first_serve_percentage": winner_stats['fs_perc'],
                "loser_first_serve_percentage": loser_stats['fs_perc'],
                "winner_first_serve_points_won_percentage": winner_stats['fs_pts_won_perc'],
                "loser_first_serve_points_won_percentage": loser_stats['fs_pts_won_perc'],
                "winner_second_serve_points_won_percentage": winner_stats['ss_pts_won_perc'],
                "loser_second_serve_points_won_percentage": loser_stats['ss_pts_won_perc'],
                "winner_return_points_won_percentage": winner_stats['ret_pts_won_perc'],
                "loser_return_points_won_percentage": loser_stats['ret_pts_won_perc'],
                "winner_aces": winner_stats['aces'],
                "loser_aces": loser_stats['aces'],
                "winner_unforced_errors": winner_stats['ue'],
                "loser_unforced_errors": loser_stats['ue'],
                "match_id": i
            })
        return pd.DataFrame(data)

    def get_player_stats(self, player_name_raw: str) -> dict:
        """Retrieves aggregated stats for a player, uses cache."""
        normalized_player_name = _normalize_player_name(player_name_raw)

        if normalized_player_name in self.player_stats_cache:
            return self.player_stats_cache[normalized_player_name]

        winner_matches = self.match_data[self.match_data['winner_name'] == normalized_player_name]
        loser_matches = self.match_data[self.match_data['loser_name'] == normalized_player_name]

        if winner_matches.empty and loser_matches.empty:
            return {}

        stat_cols = ['first_serve_percentage', 'first_serve_points_won_percentage',
                     'second_serve_points_won_percentage', 'return_points_won_percentage',
                     'aces', 'unforced_errors']

        player_stats_list = []

        for _, row in winner_matches.iterrows():
            stats = {col: row.get(f'winner_{col}', np.nan) for col in stat_cols}
            player_stats_list.append(stats)

        for _, row in loser_matches.iterrows():
            stats = {col: row.get(f'loser_{col}', np.nan) for col in stat_cols}
            player_stats_list.append(stats)

        if not player_stats_list:
            return {}

        all_player_stats_df = pd.DataFrame(player_stats_list)
        all_player_stats = all_player_stats_df.mean(skipna=True).to_dict()

        all_player_stats["win_loss_record"] = f"{len(winner_matches)}-{len(loser_matches)}"
        all_player_stats["matches_played"] = len(winner_matches) + len(loser_matches)
        all_player_stats["current_rank"] = self.atp_scraper.get_player_rank(player_name_raw)
        self.player_stats_cache[normalized_player_name] = all_player_stats
        return self.player_stats_cache.get(normalized_player_name, {})

    def get_all_player_names(self) -> list:
        # Collect normalized historical names, then convert to Title Case for display
        historical_players_set_normalized = set(self.match_data['winner_name'].tolist() + self.match_data['loser_name'].tolist()) if not self.match_data.empty else set()

        # We need a way to map normalized names back to a "common" display name.
        # For historical players, we'll try to find an original name from scraper data if possible,
        # otherwise, we'll title case the normalized name.
        all_display_names = set()
        for p_info in self.atp_scraper.player_rankings:
            all_display_names.add(p_info['original_name'])

        for normalized_h_name in historical_players_set_normalized:
            found_match = False
            for p_info in self.atp_scraper.player_rankings:
                if normalized_h_name == p_info['name']:
                    all_display_names.add(p_info['original_name'])
                    found_match = True
                    break
            if not found_match:
                all_display_names.add(normalized_h_name.title()) # Fallback to title casing

        return sorted(list(all_display_names))

    def get_match_data_for_training(self) -> pd.DataFrame:
        df = self.match_data.copy()
        stat_cols = ['first_serve_percentage', 'first_serve_points_won_percentage',
                     'second_serve_points_won_percentage', 'return_points_won_percentage',
                     'aces', 'unforced_errors']

        critical_percentage_cols = [f'{p}_{s}' for p in ['winner', 'loser'] for s in ['first_serve_percentage', 'first_serve_points_won_percentage', 'second_serve_points_won_percentage', 'return_points_won_percentage']]
        df_cleaned = df.dropna(subset=critical_percentage_cols)

        if df_cleaned.empty:
            print("Warning: No complete match data available after cleaning for training. This might affect model training quality.")
            return pd.DataFrame()

        df_p1_winner = df_cleaned.assign(player1_wins=1, player1_name=df_cleaned['winner_name'], player2_name=df_cleaned['loser_name'])
        df_p1_loser = df_cleaned.assign(player1_wins=0, player1_name=df_cleaned['loser_name'], player2_name=df_cleaned['winner_name'])

        for col in stat_cols:
            df_p1_winner[f'player1_{col}'] = df_p1_winner[f'winner_{col}']
            df_p1_winner[f'player2_{col}'] = df_p1_winner[f'loser_{col}']
            df_p1_loser[f'player1_{col}'] = df_p1_loser[f'loser_{col}']
            df_p1_loser[f'player2_{col}'] = df_p1_loser[f'winner_{col}']

        training_cols = ['player1_name', 'player2_name', 'player1_wins'] + \
                         [f'player1_{c}' for c in stat_cols] + \
                         [f'player2_{c}' for c in stat_cols]

        return pd.concat([df_p1_winner[training_cols], df_p1_loser[training_cols]], ignore_index=True)


## 5. Match prediction model

Engineers player-to-player difference features, trains a logistic regression classifier, evaluates it on a held-out test split, and produces matchup probabilities.


In [ ]:
# --- 2. Match Prediction Model ---
class MatchPredictor:
    def __init__(self):
        self.model = None
        self.features = [
            "first_serve_percentage_diff",
            "first_serve_points_won_percentage_diff",
            "second_serve_points_won_percentage_diff",
            "return_points_won_percentage_diff",
            "aces_diff",
            "unforced_errors_diff"
        ]

    def train_model(self, df: pd.DataFrame):
        print("Training prediction model...")

        if df.empty:
            print("Cannot train model: Input DataFrame is empty.")
            self.model = None
            return

        for feature_base in ['first_serve_percentage', 'first_serve_points_won_percentage',
                             'second_serve_points_won_percentage', 'return_points_won_percentage',
                             'aces']:
            df[f'{feature_base}_diff'] = df[f'player1_{feature_base}'] - df[f'player2_{feature_base}']

        df['unforced_errors_diff'] = df['player2_unforced_errors'] - df['player1_unforced_errors']

        X = df[self.features].fillna(0)
        y = df['player1_wins']

        if len(X) < 2 or X.isnull().values.any() or y.isnull().any():
            print("Not enough valid data after feature engineering to train the model. Need at least 2 samples and no NaNs in features/target.")
            self.model = None
            return
        if len(y.unique()) < 2:
            print("Cannot train model: Target variable 'player1_wins' contains only one class. Ensure your data has both wins and losses.")
            self.model = None
            return

        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

        self.model = LogisticRegression(random_state=42, solver='liblinear').fit(X_train, y_train)
        print(f"Model training complete. Accuracy: {accuracy_score(y_test, self.model.predict(X_test)):.2f}")

    def predict_match_outcome(self, p1_stats: dict, p2_stats: dict, p1_name: str, p2_name: str) -> str:
        if self.model is None: return "Prediction model not trained. Type 'train model'."
        if not p1_stats or not p2_stats: return "Cannot predict: Missing stats for one or both players."

        features_input = {}
        for feature_base in ['first_serve_percentage', 'first_serve_points_won_percentage',
                             'second_serve_points_won_percentage', 'return_points_won_percentage',
                             'aces']:
            features_input[f'{feature_base}_diff'] = p1_stats.get(feature_base, 0) - p2_stats.get(feature_base, 0)

        features_input['unforced_errors_diff'] = p2_stats.get('unforced_errors', 0) - p1_stats.get('unforced_errors', 0)

        pred_df = pd.DataFrame([features_input])

        try:
            pred_df = pred_df[self.features]
        except KeyError as e:
            return f"Error preparing prediction features: {e}. One or more required stats might be missing for players."

        pred_df = pred_df.fillna(0)

        if pred_df.shape[1] != len(self.features):
            return f"Error: Mismatch in prediction features. Expected {len(self.features)}, got {pred_df.shape[1]}."

        try:
            proba = self.model.predict_proba(pred_df)[0]
            p1_prob, p2_prob = proba[1], proba[0]

            winner, w_prob, loser, l_prob = (p1_name, p1_prob, p2_name, p2_prob) if p1_prob > p2_prob else (p2_name, p2_prob, p1_name, p1_prob)
            return (f"Predicted Winner: **{winner}** (Probability: {w_prob:.2%})\n"
                    f"Against {loser} (Probability: {l_prob:.2%})")
        except Exception as e:
            return f"Error during prediction calculation: {e}. This might indicate an issue with the model or input data."


## 6. Interactive chatbot

Provides commands for viewing player stats, listing players, training the model, and requesting matchup predictions.


In [ ]:
# --- 3. Chatbot Interface ---
class TennisChatbot:
    def __init__(self):
        self.atp_scraper = ATPScraper()
        # Ensure rankings are scraped immediately upon init for initial use
        self.atp_scraper.scrape_player_rankings(num_players=100, force_scrape=True)
        self.data_loader = TennisDataLoader(self.atp_scraper)
        self.predictor = MatchPredictor()
        self.trained = False

    def process_command(self, command: str) -> str:
        cmd = command.lower().strip()
        if "hello" in cmd or "hi" in cmd: return "Hello my name is Bill! I provide tennis stats and predictions. How can I help?"
        elif "stats for" in cmd:
            player_name_raw = cmd.replace("stats for", "").strip()
            return self._get_player_stats_response(player_name_raw)
        elif "list players" in cmd or "who are the players" in cmd: return self._list_players_response()
        elif "train model" in cmd: return self._train_model_response()
        elif "predict match" in cmd: return "To predict, use 'predict [Player1] vs [Player2]'."
        elif "predict" in cmd and "vs" in cmd: return self._handle_prediction_request(cmd)
        elif "help" in cmd: return self._help_message()
        elif "exit" in cmd or "bye" in cmd: return "Goodbye! See you on the court."
        return "I'm not sure how to handle that. Try 'help'."

    def _get_player_stats_response(self, player_name_raw: str) -> str:
        stats = self.data_loader.get_player_stats(player_name_raw)

        display_player_name = player_name_raw.title()

        if not stats:
            return f"Sorry, no historical stats found for **{display_player_name}**. This could be because there's no data for them in the historical dataset, or their name might be spelled differently. Try 'list players' for available names."

        response = f"Stats for **{display_player_name}** (historical data):\n"

        rank_info = stats.get("current_rank")
        if rank_info is not None:
            response += f"- **Current Rank**: {rank_info}\n"
        else:
            response += f"- **Current Rank**: Not available (Could not retrieve live ranking for this player. Try slightly different spelling or or a player from 'list players').\n"

        stat_keys_to_display = [
            'first_serve_percentage', 'first_serve_points_won_percentage',
            'second_serve_points_won_percentage', 'return_points_won_percentage',
            'aces', 'unforced_errors', 'win_loss_record', 'matches_played'
        ]

        for k in stat_keys_to_display:
            display_name = k.replace('_', ' ').title()
            value = stats.get(k)

            if k == 'unforced_errors':
                response += f"- {display_name}: {int(round(value))} (approx.)\n" if pd.notna(value) else f"- {display_name}: Data not available, using simulated data for predictions.\n"
            elif pd.isna(value) or value is None:
                response += f"- {display_name}: Data not available from source, using simulated data for predictions.\n"
            elif 'percentage' in k or 'perc' in k:
                response += f"- {display_name}: {value:.2%}\n"
            elif isinstance(value, float):
                response += f"- {display_name}: {int(round(value))}\n"
            else:
                response += f"- {display_name}: {value}\n"
        return response

    def _list_players_response(self) -> str:
        live_players_info = [{"name": p['original_name'], "rank": p['rank']} for p in self.atp_scraper.player_rankings[:20]]

        # Get all historical names, converting them back to a presentable Title Case
        historical_players_set_normalized = set(self.data_loader.match_data['winner_name'].tolist() + self.data_loader.match_data['loser_name'].tolist()) if not self.data_loader.match_data.empty else set()

        all_display_names = set()
        # Add live player original names (which are already display-friendly)
        for p_info in self.atp_scraper.player_rankings:
            all_display_names.add(p_info['original_name'])

        # Add historical player names. Prioritize matching with scraped original names
        # for consistency, otherwise title case the normalized name.
        for normalized_h_name in historical_players_set_normalized:
            found_match = False
            for p_info in self.atp_scraper.player_rankings:
                if normalized_h_name == p_info['name']: # Check if normalized historical matches normalized live
                    all_display_names.add(p_info['original_name']) # Add the original live name
                    found_match = True
                    break
            if not found_match:
                # If no match with live names, just title case the normalized historical name
                all_display_names.add(normalized_h_name.title())

        sorted_all_display_names = sorted(list(all_display_names))

        response_parts = []
        if live_players_info:
            response_parts.append("Players (top 20 live rankings first):\n" + "".join([f"- **{p['name']}** (Rank: {p['rank']})\n" for p in live_players_info]))

        # Filter historical players to only include those not already in the top 20 live list
        live_player_names_normalized = {_normalize_player_name(p['name']) for p in live_players_info}
        other_historical_players_display_filtered = sorted([p for p in sorted_all_display_names if _normalize_player_name(p) not in live_player_names_normalized])

        if other_historical_players_display_filtered:
            response_parts.append("\nAnd historical data for others including: " + ", ".join(other_historical_players_display_filtered[:10]) + ("..." if len(other_historical_players_display_filtered) > 10 else ""))

        return "".join(response_parts) if response_parts else "No player data available. This might be due to an issue loading historical data or scraping live rankings."


    def _train_model_response(self) -> str:
        if self.trained: return "Model already trained."
        if self.data_loader.match_data.empty: return "Cannot train: No historical data loaded. Try checking the data source or using simulated data."
        try:
            training_data = self.data_loader.get_match_data_for_training()
            if training_data.empty:
                return "Cannot train: Prepared training data is empty after cleaning. This could mean critical columns are missing or all rows were dropped."
            self.predictor.train_model(training_data)
            self.trained = True
            return "Prediction model trained successfully!"
        except Exception as e:
            return f"Error during training: {e}. Ensure enough data is available and features can be calculated."

    def _handle_prediction_request(self, cmd: str) -> str:
        if not self.trained: return "Please train the model first ('train model')."
        try:
            players_str = cmd.replace("predict", "").replace("match", "").strip()
            if "vs" not in players_str:
                return "Invalid prediction format. Use 'predict [Player1] vs [Player2]'."

            p1_name_raw, p2_name_raw = [p.strip() for p in players_str.split("vs", 1)]

            p1_stats, p2_stats = self.data_loader.get_player_stats(p1_name_raw), self.data_loader.get_player_stats(p2_name_raw)

            missing_players = []
            if not p1_stats:
                missing_players.append(p1_name_raw.title())
            if not p2_stats:
                missing_players.append(p2_name_raw.title())

            if missing_players:
                return f"Missing historical stats for: {', '.join(missing_players)}. Try 'list players' to see available players."

            return self.predictor.predict_match_outcome(p1_stats, p2_stats, p1_name_raw.title(), p2_name_raw.title())
        except Exception as e:
            return f"Error predicting: {e}. Please ensure player names are correct and the format is 'predict [Player1] vs [Player2]'."

    def _help_message(self) -> str:
        return ("Commands:\n- '**Hello**' or '**Hi**': Greet the bot.\n- '**Stats for [Player Name]**': Get historical statistics for a player.\n- '**List players**' or '**Who are the players**': See a list of top-ranked players and others with historical data.\n"
                "- '**Train model**': Train the match prediction model using historical data.\n- '**Predict [Player1] vs [Player2]**': Predict the winner of a match between two players (model must be trained).\n- '**Help**': Display this help message.\n- '**Exit**' or '**Bye**': End the conversation.")


## 7. Run the chatbot

Starts the command-line interaction loop. Run this cell after the earlier cells.


In [ ]:
# --- Main Interaction Loop ---
def run_chatbot():
    print("Initializing Tennis Chatbot...")
    chatbot = TennisChatbot()
    print("\nBot: " + chatbot.process_command("hello"))
    print("\nBot: " + chatbot.process_command("help"))
    while True:
        user_input = input("\nYou: ")
        response = chatbot.process_command(user_input)
        print(f"Bot: {response}")
        if "goodbye" in response.lower(): break

if __name__ == "__main__":
    run_chatbot()
